# Demo del Deliverable 3 · sistema de agentes con Phi-3.5-mini

Corre en vivo los tres casos que eligió la regla de `PROTOCOLO_D3.md` y muestra cada mensaje
entre agentes y herramientas. Cada salida del modelo se compara con la corrida final guardada
en `trazas/`. Con generación greedy tiene que decir `= guardada`.

Requiere GPU T4. Las celdas 1 a 3 preparan el entorno, la 4 corre los tres casos, la 5 una
pregunta escrita en el momento y la 6 baja la traza de esa pregunta.

In [ ]:
# 1 · repositorio y dependencias
# El commit de la demo: los prompts, los agentes y las herramientas son los de la corrida final
# (9158550), y trae scripts/demo_d3.py con la pregunta en vivo, que guarda su traza.
# Con '' clona main.
COMMIT = '75ed9b2'

%cd /content
!pip -q install transformers==5.17.0 accelerate==1.14.0 bitsandbytes==0.50.2
!rm -rf /content/proyecto && git clone -q https://github.com/Luquitas02/genai-inscripcion-ramos /content/proyecto
import subprocess
if COMMIT:
    subprocess.run(['git', '-C', '/content/proyecto', 'checkout', '-q', COMMIT], check=True)
!git -C /content/proyecto log -1 --format='commit %h  %ad  %s'
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# 2 · autopruebas: la demo reproduce las trazas guardadas sin usar la GPU
import sys, os
os.chdir('/content/proyecto/scripts')
sys.path.insert(0, '/content/proyecto/scripts')
r = subprocess.run([sys.executable, 'demo_d3.py', '--pruebas'], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
r = subprocess.run([sys.executable, 'runner_d3.py', '--pruebas'], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])

In [ ]:
# 3 · carga de Phi-3.5-mini (tres a cinco minutos)
import runner_d3, demo_d3
modelo = runner_d3.cargar_modelo('microsoft/Phi-3.5-mini-instruct')
print('modelo cargado · revisión', getattr(modelo.modelo.config, '_commit_hash', '?'))

In [ ]:
# 4 · los tres casos del protocolo, en vivo
demo_d3.video(modelo)

In [ ]:
# 5 · una pregunta escrita en el momento, sobre el historial del redactado 0
# No tiene etiqueta: se ve la decisión del sistema, sin acierto ni falla.
PREGUNTA = '¿puedo tomar estadística este semestre si ya pasé cálculo 2?'
demo_d3.en_vivo(modelo, PREGUNTA)

In [ ]:
# 6 · baja la traza y la fila de la pregunta en vivo
# Van a trazas/ y resultados/ del repositorio.
from google.colab import files
files.download('/content/proyecto/trazas/en_vivo.jsonl')
files.download('/content/proyecto/resultados/en_vivo.raw.jsonl')